# 03 · Work with a multi-variable Dataset

An xarray `Dataset` can hold several aligned cubes. Select the variable whose
meaning matches a verb, run separate pipelines, and bring the results together
in one explanatory figure.

**Pattern:** `Dataset` → select a `DataArray` → one pipeline per scientific question.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe, verbs as v

# A fixed seed makes the example exactly repeatable while still looking like
# measurements with natural variation.
rng = np.random.default_rng(19)
time = pd.date_range("2024-04-01", periods=20, freq="D")
y = np.linspace(41.0, 40.0, 4)
x = np.linspace(-106.0, -105.0, 5)
temperature = 18 + np.linspace(0, 7, time.size)[:, None, None] + rng.normal(0, 1, (20, 4, 5))
precipitation = rng.gamma(1.4, 2.2, size=(20, 4, 5))

# A Dataset is a labeled collection of aligned DataArrays. Each variable has
# its own units but shares the same time/y/x coordinate system.
dataset = xr.Dataset(
    {
        "temperature": (("time", "y", "x"), temperature, {"units": "degC"}),
        "precipitation": (("time", "y", "x"), precipitation, {"units": "mm day-1"}),
    },
    coords={"time": time, "y": y, "x": x},
    attrs={"source": "deterministic multi-variable example"},
)

# Select one variable before applying a verb whose scientific meaning matches
# it. anomaly preserves the cube shape while subtracting each pixel's mean.
temperature_anomaly = (
    pipe(dataset["temperature"])
    | v.anomaly(dim="time")
).unwrap()

# Reducing both spatial axes produces one regional value for every date.
# keep_dim=False intentionally returns a 1D time series rather than a 3D cube.
regional_precipitation = (
    pipe(dataset["precipitation"])
    | v.mean(dim=("y", "x"), keep_dim=False)
).unwrap()

assert temperature_anomaly.dims == ("time", "y", "x")
assert regional_precipitation.dims == ("time",)

# The side-by-side views answer different questions from the same Dataset.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), constrained_layout=True)
temperature_anomaly.isel(time=-1).plot(
    ax=axes[0], cmap="RdBu_r", center=0, cbar_kwargs={"label": "°C anomaly"}
)
axes[0].set_title("Latest temperature anomaly")
regional_precipitation.plot(ax=axes[1], marker="o", color="#2f6267")
axes[1].set_title("Spatial mean precipitation")
axes[1].set_ylabel("mm day⁻¹")
plt.show()